*Import Library*

In [ ]:
# Import library
import pandas as pd
import numpy as np
!pip install google-play-scraper
from google_play_scraper import app, Sort, reviews_all, reviews
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
import saka

In [ ]:

pd.options.mode.chained_assignment = None  # Menonaktifkan peringatan chaining
seed = 0
np.random.seed(seed)  # Mengatur seed untuk reproduktibilitas
import matplotlib.pyplot as plt  # Matplotlib untuk visualisasi data
import seaborn as sns  # Seaborn untuk visualisasi data statistik, mengatur gaya visualisasi
from sklearn.metrics import accuracy_score
 
from nltk.tokenize import word_tokenize  # Tokenisasi teks
from nltk.corpus import stopwords  # Daftar kata-kata berhenti dalam teks
 
!pip install sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory  # Stemming (penghilangan imbuhan kata) dalam bahasa Indonesia
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory  # Menghapus kata-kata berhenti dalam bahasa Indonesia
 
from wordcloud import WordCloud  # Membuat visualisasi berbentuk awan kata (word cloud) dari teks
 
import nltk  # Import pustaka NLTK (Natural Language Toolkit).
nltk.download('punkt')  # Mengunduh dataset yang diperlukan untuk tokenisasi teks.
nltk.download('stopwords')  # Mengunduh dataset yang berisi daftar kata-kata berhenti (stopwords) dalam berbagai bahasa.

[nltk_data] Downloading package punkt to /home/codespace/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

*Mengambil data dari Google Play Store*

In [4]:
# Mengambil semua ulasan dari aplikasi dengan ID 'com.mobile.legends&hl' di Google Play Store.
pd.options.mode.chained_assignment = None 
aplikasi_id = 'com.miHoYo.GenshinImpact'
scrapreview = reviews_all(
    aplikasi_id,            # ID Aplikasi target
    lang='id',                # Bahasa ulasan: Indonesia
    country='id',             # Negara: Indonesia
    sort=Sort.NEWEST,         # Mengambil ulasan terbaru agar datanya variatif
    count=10000               # TARGET: 10.000 data
)


*Menyimpan data pada file -ulasan_aplikasi.csv- *

In [5]:
# Menyimpan ulasan dalam file CSV
df_asli = pd.DataFrame(scrapreview)

df_asli.to_csv("ulasan_aplikasi.csv", index= False)

# Menghitung jumlah baris dan kolom
jumlah_ulasan, jumlah_kolom = df_asli.shape
jumlah_ulasan, jumlah_kolom

# Menampilkan hasil
df_asli.head()
df_asli.info()

<class 'pandas.DataFrame'>
RangeIndex: 225711 entries, 0 to 225710
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   reviewId              225711 non-null  str           
 1   userName              225711 non-null  str           
 2   userImage             225711 non-null  str           
 3   content               225711 non-null  str           
 4   score                 225711 non-null  int64         
 5   thumbsUpCount         225711 non-null  int64         
 6   reviewCreatedVersion  145799 non-null  str           
 7   at                    225711 non-null  datetime64[us]
 8   replyContent          16841 non-null   str           
 9   repliedAt             16841 non-null   datetime64[us]
 10  appVersion            145799 non-null  str           
dtypes: datetime64[us](2), int64(2), str(7)
memory usage: 78.9 MB


*Mengambil kolom content dan score sebagai data yang akan digunakan*

In [6]:
# Memilih hanya kolom 'content' dan 'score' untuk analisis lebih lanjut
df_select = df_asli[['content', 'score']]
df_select.info()

<class 'pandas.DataFrame'>
RangeIndex: 225711 entries, 0 to 225710
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   content  225711 non-null  str  
 1   score    225711 non-null  int64
dtypes: int64(1), str(1)
memory usage: 16.3 MB


*Mengatasi data yang tidak relevan*

In [7]:
df_final = df_select.dropna()
df_final = df_select.drop_duplicates()
# Menampilkan hasil
df_final.head()
df_final.info()

<class 'pandas.DataFrame'>
Index: 171898 entries, 0 to 225710
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   content  171898 non-null  str  
 1   score    171898 non-null  int64
dtypes: int64(1), str(1)
memory usage: 16.4 MB


In [8]:
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
 
def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # menghapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # menghapus hashtag
    text = re.sub(r'RT[\s]', '', text) # menghapus RT
    text = re.sub(r"http\S+", '', text) # menghapus link
    text = re.sub(r'[0-9]+', '', text) # menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka
 
    text = text.replace('\n', ' ') # mengganti baris baru dengan spasi
    text = text.translate(str.maketrans('', '', string.punctuation)) # menghapus semua tanda baca
    text = text.strip(' ') # menghapus karakter spasi dari kiri dan kanan teks
    return text
 
def casefoldingText(text): # Mengubah semua karakter dalam teks menjadi huruf kecil
    text = text.lower()
    return text
 
def tokenizingText(text): # Memecah atau membagi string, teks menjadi daftar token
    text = word_tokenize(text)
    return text
 
def filteringText(text): # Menghapus stopwords dalam teks
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords1 = set(stopwords.words('english'))
    listStopwords.update(listStopwords1)
    listStopwords.update(['iya','yaa','gak','nya','na','sih','ku',"di","ga","ya","gaa","loh","kah","woi","woii","woy"])
    filtered = []
    for txt in text:
        if txt not in listStopwords:
            filtered.append(txt)
    text = filtered
    return text
 
def stemmingText(text): # Mengurangi kata ke bentuk dasarnya yang menghilangkan imbuhan awalan dan akhiran atau ke akar kata
    # Membuat objek stemmer
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()
 
    # Memecah teks menjadi daftar kata
    words = text.split()
 
    # Menerapkan stemming pada setiap kata dalam daftar
    stemmed_words = [stemmer.stem(word) for word in words]
 
    # Menggabungkan kata-kata yang telah distem
    stemmed_text = ' '.join(stemmed_words)
 
    return stemmed_text
 
def toSentence(list_words): # Mengubah daftar kata menjadi kalimat
    sentence = ' '.join(word for word in list_words)
    return sentence

In [9]:

def normalize_texts(text):
    return [saka.normalize(t) for t in text]

In [10]:
# Cleaning
df_final['text_clean'] = df_final['content'].apply(cleaningText)

# Case folding
df_final['text_casefoldingText'] = df_final['text_clean'].apply(casefoldingText)

# Normalize slang
df_final['normalize_text'] = normalize_texts(df_final['text_casefoldingText'])

# Tokenizing
for res, pkg in (("tokenizers/punkt","punkt"), ("tokenizers/punkt_tab","punkt_tab")):
    try:
        nltk.data.find(res)
    except LookupError:
        nltk.download(pkg, quiet=True)

df_final['text_tokenizingText'] = df_final['normalize_text'].apply(tokenizingText)

# Stopword removal
df_final['text_stopword'] = df_final['text_tokenizingText'].apply(filteringText)

# To sentence
df_final['text_akhir'] = df_final['text_stopword'].apply(toSentence)

# Cek hasil
df_final.info()


<class 'pandas.DataFrame'>
Index: 171898 entries, 0 to 225710
Data columns (total 8 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   content               171898 non-null  str   
 1   score                 171898 non-null  int64 
 2   text_clean            171898 non-null  str   
 3   text_casefoldingText  171898 non-null  str   
 4   normalize_text        171898 non-null  str   
 5   text_tokenizingText   171898 non-null  object
 6   text_stopword         171898 non-null  object
 7   text_akhir            171898 non-null  str   
dtypes: int64(1), object(2), str(5)
memory usage: 67.1+ MB


In [11]:
def tentukan_label(score):
    if score <= 2: return 'Negatif'
    elif score == 3: return 'Netral'
    else: return 'Positif'

df_final['sentiment'] = df_final['score'].apply(tentukan_label)

df_final

,content,score,text_clean,text_casefoldingText,normalize_text,text_tokenizingText,text_stopword,text_akhir,sentiment
0,tambahkan tombol skip,1,tambahkan tombol skip,tambahkan tombol skip,tambahkan tombol skip,"[tambahkan, tombol, skip]","[tambahkan, tombol, skip]",tambahkan tombol skip,Negatif
1,PELITT,1,PELITT,pelitt,pelitt,[pelitt],[pelitt],pelitt,Negatif
2,aku main ini ngak pernah dapet karakter baru d...,2,aku main ini ngak pernah dapet karakter baru d...,aku main ini ngak pernah dapet karakter baru d...,aku main ini ngak pernah dapat karakter baru d...,"[aku, main, ini, ngak, pernah, dapat, karakter...","[main, ngak, karakter, game]",main ngak karakter game,Negatif
3,gak bisa masuk cokkkk,1,gak bisa masuk cokkkk,gak bisa masuk cokkkk,gak bisa masuk cokkkk,"[gak, bisa, masuk, cokkkk]","[masuk, cokkkk]",masuk cokkkk,Negatif
4,"benci bgt sm gi 6.6, biar apesi lu bikin gua k...",1,benci bgt sm gi biar apesi lu bikin gua kalah...,benci bgt sm gi biar apesi lu bikin gua kalah...,benci banget sm gi biar apesi kamu bikin saya ...,"[benci, banget, sm, gi, biar, apesi, kamu, bik...","[benci, banget, sm, gi, biar, apesi, bikin, ka...",benci banget sm gi biar apesi bikin kalah rate...,Negatif
...,...,...,...,...,...,...,...,...,...
225703,Di hp gue gak ada kendala buat mainin genshin ...,5,Di hp gue gak ada kendala buat mainin genshin ...,di hp gue gak ada kendala buat mainin genshin ...,di hp saya gak ada kendala buat mainin genshin...,"[di, hp, saya, gak, ada, kendala, buat, mainin...","[hp, kendala, mainin, genshin, impact]",hp kendala mainin genshin impact,Positif
225705,MANTAP AJG!!,5,MANTAP AJG,mantap ajg,mantap ajg,"[mantap, ajg]","[mantap, ajg]",mantap ajg,Positif
225706,"Saya sangat menantikan game ini, semoga saya b...",5,Saya sangat menantikan game ini semoga saya bi...,saya sangat menantikan game ini semoga saya bi...,saya sangat menantikan game ini semoga saya bi...,"[saya, sangat, menantikan, game, ini, semoga, ...","[game, semoga, memainkannya, perangkat, hppc, ...",game semoga memainkannya perangkat hppc miliki...,Positif
225707,Wah gamenya bagus banget Udh open world grafik...,5,Wah gamenya bagus banget Udh open world grafik...,wah gamenya bagus banget udh open world grafik...,wah gamenya bagus banget udh open world grafik...,"[wah, gamenya, bagus, banget, udh, open, world...","[gamenya, bagus, banget, udh, open, world, gra...",gamenya bagus banget udh open world grafik bag...,Positif


*Menggunakan fitur Ekstraksi TF-IDF*

In [12]:

# Pisahkan data menjadi fitur (tweet) dan label (sentimen)
X = df_final['text_akhir']
y = df_final['sentiment']

# Ekstraksi fitur dengan TF-IDF
tfidf = TfidfVectorizer(max_features=200, min_df=17, max_df=0.8 )
X_tfidf = tfidf.fit_transform(X)

# Konversi hasil ekstraksi fitur menjadi dataframe
features_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())
 
# Menampilkan hasil ekstraksi fitur
features_df
 
 # Bagi data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(features_df, y, test_size=0.2, random_state=42)

*Modeling Menggunakan Naive Bayes*

In [13]:
from sklearn.naive_bayes import BernoulliNB
 
# Membuat objek model Naive Bayes (Bernoulli Naive Bayes)
naive_bayes = BernoulliNB()
 
# Melatih model Naive Bayes pada data pelatihan
naive_bayes.fit(X_train, y_train)
 
# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_nb = naive_bayes.predict(X_train)
y_pred_test_nb = naive_bayes.predict(X_test)
 
# Evaluasi akurasi model Naive Bayes
accuracy_train_nb = accuracy_score(y_pred_train_nb, y_train)
accuracy_test_nb = accuracy_score(y_pred_test_nb, y_test)
 
# Menampilkan akurasi
print('Naive Bayes - accuracy_train:', accuracy_train_nb)
print('Naive Bayes - accuracy_test:', accuracy_test_nb)

Naive Bayes - accuracy_train: 0.7592824212103143
Naive Bayes - accuracy_test: 0.754653868528214


*Modeling Menggunakan Random Forest*

In [14]:
from sklearn.ensemble import RandomForestClassifier
 
# Membuat objek model Random Forest
random_forest = RandomForestClassifier()
 
# Melatih model Random Forest pada data pelatihan
random_forest.fit(X_train, y_train)
 
# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_rf = random_forest.predict(X_train)
y_pred_test_rf = random_forest.predict(X_test)
 
# Evaluasi akurasi model Random Forest
accuracy_train_rf = accuracy_score(y_pred_train_rf, y_train)
accuracy_test_rf = accuracy_score(y_pred_test_rf, y_test)
 
# Menampilkan akurasi
print('Random Forest - accuracy_train:', accuracy_train_rf)
print('Random Forest - accuracy_test:', accuracy_test_rf)

Random Forest - accuracy_train: 0.8806847103651886
Random Forest - accuracy_test: 0.7707097149505526


In [15]:
from sklearn.linear_model import LogisticRegression
 
# Membuat objek model Logistic Regression
logistic_regression = LogisticRegression()
 
# Melatih model Logistic Regression pada data pelatihan
logistic_regression.fit(X_train, y_train)
 
# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_lr = logistic_regression.predict(X_train)
y_pred_test_lr = logistic_regression.predict(X_test)
 
# Evaluasi akurasi model Logistic Regression pada data pelatihan
accuracy_train_lr = accuracy_score(y_pred_train_lr, y_train)
 
# Evaluasi akurasi model Logistic Regression pada data uji
accuracy_test_lr = accuracy_score(y_pred_test_lr, y_test)
 
# Menampilkan akurasi
print('Logistic Regression - accuracy_train:', accuracy_train_lr)
print('Logistic Regression - accuracy_test:', accuracy_test_lr)


Logistic Regression - accuracy_train: 0.7782399394988292
Logistic Regression - accuracy_test: 0.7742292030250145


In [16]:
from sklearn.tree import DecisionTreeClassifier
 
# Membuat objek model Decision Tree
decision_tree = DecisionTreeClassifier()
 
# Melatih model Decision Tree pada data pelatihan
decision_tree.fit(X_train, y_train)
 
# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_dt = decision_tree.predict(X_train)
y_pred_test_dt = decision_tree.predict(X_test)
 
# Evaluasi akurasi model Decision Tree
accuracy_train_dt = accuracy_score(y_pred_train_dt, y_train)
accuracy_test_dt = accuracy_score(y_pred_test_dt, y_test)
 
# Menampilkan akurasi
print('Decision Tree - accuracy_train:', accuracy_train_dt)
print('Decision Tree - accuracy_test:', accuracy_test_dt)

Decision Tree - accuracy_train: 0.8806919821405198
Decision Tree - accuracy_test: 0.7385980221058756


In [18]:
final_tfidf = logistic_regression
final_tfidf

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul